<a href="https://colab.research.google.com/github/maierav/ai_oscp_neuro/blob/main/notebooks/oddball_confirmatory_ecephys.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Feature-oddball prediction error — confirmatory analysis (Neuropixels)

This notebook reproduces the **confirmatory** feature-oddball result across the 9 CCF-labelled
standard-oddball ecephys sessions in DANDI **001637** (OpenScope Community Predictive Processing).
It extends the single-session example (`oddball_prediction_error_ecephys.ipynb`) to the full cohort
and adds the population, laminar, and error-type analyses.

**What it computes, per session, in a single streaming pass:**
1. **Deviance index** `DvI = (R_oddball − R_control) / (|R_oddball| + |R_control|)` — the adaptation-controlled
   contrast against the physically-identical grating from the equiprobable `Control block 1`.
2. **Tuning** (OSI, preferred orientation, split-half reliability) from the control-block drifting sweep.
3. **PSTHs** for standard / oddball / control (−100→+500 ms, 10 ms bins).
4. **Omission response** — firing when an expected grating is withheld (tuning-free, stimulus-free error).

**Pre-registered design** (`docs/oddball_analysis_plan.md`): the primary outcome is DvI against the
many-standards control, *not* the naive oddball-vs-standard index (OI), because OI is inflated by
stimulus-specific adaptation of the frequent standard.


In [ ]:
#@title Install dependencies
!pip -q install pynwb h5py remfile requests pandas numpy matplotlib scipy statsmodels

In [ ]:
#@title Streaming + single-pass extraction
import numpy as np, pandas as pd, h5py, remfile, requests, re
from scipy import stats as ss

def s3_url(dandiset, asset_id, version="draft"):
    """Resolve a DANDI asset to a presigned S3 URL (anonymous, no credentials)."""
    u=f"https://api.dandiarchive.org/api/dandisets/{dandiset}/versions/{version}/assets/{asset_id}/download/"
    r=requests.get(u,allow_redirects=False,timeout=60)
    return r.headers["Location"]

def col(g,c):
    v=g[c][:]; return np.array([x.decode() if isinstance(x,bytes) else x for x in v])

def decode_ccf(loc):
    """CCF acronym -> (area, layer). CA1/2/3 are subfields, not layers."""
    area=re.match(r"^[A-Za-z]+",loc).group(0) if re.match(r"^[A-Za-z]+",loc) else loc
    lay=re.search(r"(\d[ab]?|2/3)$",loc); return area,(lay.group(1) if lay else None)

def extract_session(subject, aid):
    """Open one NWB once; return per-unit table + PSTH tensor."""
    PRE,POST,BW=0.1,0.5,0.01; EDGES=np.arange(-PRE,POST+BW,BW); CEN=EDGES[:-1]+BW/2
    CONDS=["standard","oddball_90","control_90","oddball_45","control_45"]
    RESP=(0.045,0.295); BASE=(-0.10,-0.005); RT=(0.05,0.35); BT=(-0.25,-0.02)
    fh=h5py.File(remfile.File(s3_url("001637",aid)),"r")
    try:
        I=fh["intervals"]; gO=I["Standard mismatch block_presentations"]; gC=I["Control block 1_presentations"]
        def bt(g,ori,contrast=1.0,dmin=0.3,tt=None):
            O=g["Orientation"][:].astype(float);Cc=g["contrast"][:].astype(float);d=g["stop_time"][:]-g["start_time"][:]
            m=np.isclose(O,ori,atol=0.05)&(Cc==contrast)&(d>=dmin)
            if tt is not None: m&=(col(g,"TrialType")==tt)
            return g["start_time"][:][m]
        T={"standard":bt(gO,0.0,tt="standard"),"oddball_90":bt(gO,1.6,tt="orientation_90"),
           "control_90":bt(gC,1.6),"oddball_45":bt(gO,0.8,tt="orientation_45"),"control_45":bt(gC,0.8)}
        t_omit=bt(gO,0.0,contrast=0.0,tt="omission")
        Oc=gC["Orientation"][:].astype(float);Ccc=gC["contrast"][:].astype(float);TFc=gC["TemporalFrequency"][:].astype(float)
        tsc=gC["start_time"][:];durc=gC["stop_time"][:]-gC["start_time"][:]
        drift=(Ccc>0)&(TFc>0)&(durc>=0.3); dirs=np.unique(np.round(Oc[drift],3))
        U=fh["units"];st=U["spike_times"][:];sti=U["spike_times_index"][:]
        def spikes(i): return st[(0 if i==0 else sti[i-1]):sti[i]]
        n=len(sti); qc=U["default_qc"][:] if "default_qc" in U else np.ones(n,bool)
        E=fh["general/extracellular_ephys/electrodes"];eloc=col(E,"location");egroup=col(E,"group_name")
        dev=col(U,"device_name");eci=U["extremum_channel_index"][:]
        groups=sorted(set(egroup),key=lambda gn:np.where(egroup==gn)[0][0])
        offs={gn:int(np.where(egroup==gn)[0][0]) for gn in groups};bl={gn:int((egroup==gn).sum()) for gn in groups}
        def d2g(d):
            for gn in groups:
                if d[-1].lower() in gn.lower(): return gn
            return groups[0]
        dmap={d:d2g(d) for d in set(dev)}
        # NOTE: extremum_channel_index is PER-PROBE; offset into the stacked electrodes table
        u_loc=np.array([eloc[offs[dmap[dev[i]]]+min(int(eci[i]),bl[dmap[dev[i]]]-1)] for i in range(n)])
        vis=np.where(qc & np.array([bool(re.match("VIS",r)) for r in u_loc]))[0]
        def rate(sp,times,w): lo=np.searchsorted(sp,times+w[0]);hi=np.searchsorted(sp,times+w[1]);return (hi-lo)/(w[1]-w[0])
        def idx(a,b): A,B=np.mean(a),np.mean(b);d=abs(A)+abs(B);return (A-B)/d if d>0 else 0.0
        def unit_psth(sp,times):
            if len(times)==0: return np.zeros(len(CEN))
            lo=np.searchsorted(sp,times.min()-PRE);hi=np.searchsorted(sp,times.max()+POST);sp2=sp[lo:hi]
            if len(sp2)==0: return np.zeros(len(CEN))
            rel=(sp2[None,:]-times[:,None]).ravel();rel=rel[(rel>=-PRE)&(rel<POST)]
            return np.histogram(rel,EDGES)[0]/(len(times)*BW)
        recs=[]; P=np.zeros((len(vis),len(CONDS),len(CEN)))
        for k,u in enumerate(vis):
            sp=spikes(u)
            r={cd:rate(sp,T[cd],RESP)-rate(sp,T[cd],BASE) for cd in T}
            rp=ss.wilcoxon(r["standard"]).pvalue if np.any(r["standard"]!=0) else 1.0
            curve=np.zeros(len(dirs));h=np.zeros((2,len(dirs)))
            for j,dd in enumerate(dirs):
                tt=tsc[drift&np.isclose(np.round(Oc,3),dd)];ev=rate(sp,tt,RT)-rate(sp,tt,BT)
                curve[j]=ev.mean();h[0,j]=ev[0::2].mean() if len(ev)>1 else ev.mean();h[1,j]=ev[1::2].mean() if len(ev)>1 else ev.mean()
            rc=np.clip(curve,0,None);Ssum=rc.sum()
            if Ssum>0: osi=np.abs((rc*np.exp(2j*dirs)).sum())/Ssum; pref=(0.5*np.angle((rc*np.exp(2j*dirs)).sum()))%np.pi
            else: osi,pref=0.0,np.nan
            shr=np.corrcoef(h[0],h[1])[0,1] if np.std(h[0])>0 and np.std(h[1])>0 else 0.0
            om_ev=rate(sp,t_omit,RESP)-rate(sp,t_omit,BASE)
            area,layer=decode_ccf(u_loc[u])
            for ci_,cd in enumerate(CONDS): P[k,ci_]=unit_psth(sp,T[cd])
            recs.append(dict(subject=subject,unit=int(u),area=area,layer=layer,
                DvI_45=idx(r["oddball_45"],r["control_45"]),DvI_90=idx(r["oddball_90"],r["control_90"]),
                OI_45=idx(r["oddball_45"],r["standard"]),OI_90=idx(r["oddball_90"],r["standard"]),
                resp_p=rp,OSI=osi,pref_ori_deg=np.degrees(pref) if not np.isnan(pref) else np.nan,
                splithalf=shr,tune_peak=curve.max(),
                align90=np.cos(2*(pref-np.pi/2)) if not np.isnan(pref) else np.nan,
                om_resp=om_ev.mean(),om_p=ss.wilcoxon(om_ev).pvalue if np.any(om_ev!=0) else 1.0))
        df=pd.DataFrame(recs); df["responsive"]=df.resp_p<0.05
        return df,P,CEN,CONDS
    finally: fh.close()

print("helpers ready")

In [ ]:
#@title The 9 CCF standard-oddball sessions
SESSIONS = [
    ("830851","9b9e8abe-7b43-47f1-b8e1-4114f87898a1"),
    ("830848","228c2c2e-1daf-4bf6-9f66-eb6b2bce5ba5"),
    ("834691","9a3f32bf-ee7f-4b36-ae05-6f8217d8a1f0"),
    ("830795","1a3aac78-34f5-44c4-b644-7c16bc313ceb"),
    ("830794","a23eba88-5d08-4e4b-89e1-a2a10fd143b5"),
    ("830846","680d1c0c-e338-4d0b-ba29-4329436d2ae2"),
    ("830847","8606d35c-9c54-4b54-96ed-2252a9a54694"),
    ("830849","ce62d153-644b-4da9-b5a7-823353bb7ccb"),
    ("830852","b2f4bb1b-5092-4392-b4fa-b25802ea7769"),
]
# Each session streams ~12 GB and takes ~25-30 s. For a quick Colab pass, set N_SESSIONS=2;
# for the full confirmatory result use all 9 (~4-5 min).
N_SESSIONS = 9
SESSIONS = SESSIONS[:N_SESSIONS]
print(f"will process {len(SESSIONS)} session(s)")

In [ ]:
#@title Extract all sessions (single pass each)
import time
dfs=[]; Ps=[]; metas=[]
for subj,aid in SESSIONS:
    t0=time.time()
    df,P,CEN,CONDS=extract_session(subj,aid)
    dfs.append(df); Ps.append((subj,P))
    metas.append(dict(subject=subj,n_units=len(df),n_resp=int(df.responsive.sum())))
    print(f"  {subj}: {len(df)} units, {int(df.responsive.sum())} responsive ({time.time()-t0:.0f}s)")
MASTER=pd.concat(dfs,ignore_index=True)
G=MASTER[MASTER.responsive].copy()
G["tuned"]=(G.splithalf>0.5)&(G.tune_peak>3)
print(f"\nTOTAL: {len(MASTER)} units, {len(G)} responsive, {MASTER.subject.nunique()} sessions")

## 1 · Pooled deviance — the confirmatory result

Session-stratified bootstrap of the population median DvI. The adaptation-controlled index (DvI)
is compared to the naive oddball-vs-standard index (OI), which is inflated by adaptation.

In [ ]:
#@title Pooled DvI (session-stratified bootstrap)
def strat_boot(df,col_,nboot=5000,seed=0):
    rng=np.random.RandomState(seed); subs=df.subject.unique()
    by={s:df[df.subject==s][col_].values for s in subs}
    return np.array([np.median(np.concatenate([rng.choice(by[s],len(by[s]),replace=True) for s in subs])) for _ in range(nboot)])

for d in ["90","45"]:
    med=G[f"DvI_{d}"].median(); mb=strat_boot(G,f"DvI_{d}")
    p=ss.wilcoxon(G[f"DvI_{d}"]).pvalue
    print(f"DvI_{d}: median={med:+.3f}  95%CI[{np.percentile(mb,2.5):+.3f},{np.percentile(mb,97.5):+.3f}]  p={p:.1e}   (naive OI_{d}={G[f'OI_{d}'].median():+.3f})")

## 2 · Is the signal a tuning-sampling artifact?

Three checks: (a) does DvI depend on orientation preference? (b) do tuned and untuned units differ?
(c) does equalizing the preferred-orientation distribution change the population median?

In [ ]:
#@title Tuning-independence + sampling robustness
sub=G.dropna(subset=["align90"])
r,p=ss.pearsonr(sub.align90,sub.DvI_90)
print(f"DvI_90 ~ preference for deviant (align90): r={r:+.3f}, p={p:.2e}  (flat => not feature-specific)")
dt=G[G.tuned].DvI_90; du=G[~G.tuned].DvI_90
print(f"tuned n={len(dt)} med={dt.median():+.3f} | untuned n={len(du)} med={du.median():+.3f} | MW p={ss.mannwhitneyu(dt,du).pvalue:.3f}")
# orientation-balanced resample
Jb=G.dropna(subset=["pref_ori_deg"]).copy(); nb_=6
Jb["obin"]=np.clip((Jb.pref_ori_deg/180*nb_).astype(int),0,nb_-1); per=int(Jb.obin.value_counts().min())
rng=np.random.RandomState(0); bm=np.array([pd.concat([g.sample(per,replace=True,random_state=rng.randint(1_000_000_000)) for _,g in Jb.groupby("obin")]).DvI_90.median() for _ in range(5000)])
print(f"naive median={G.DvI_90.median():+.3f} | orientation-balanced={bm.mean():+.3f} 95%CI[{np.percentile(bm,2.5):+.3f},{np.percentile(bm,97.5):+.3f}]")

## 3 · Area × layer structure (FDR-corrected)

In [ ]:
#@title Area × layer grid
from statsmodels.stats.multitest import multipletests
Gv=G[G.area.str.startswith("VIS")&G.layer.notna()].copy()
cells=[]
for (a,l),s in Gv.groupby(["area","layer"]):
    if len(s)>=15: cells.append(dict(area=a,layer=l,n=len(s),med=s.DvI_90.median(),p=ss.wilcoxon(s.DvI_90).pvalue))
grid=pd.DataFrame(cells); grid["p_fdr"]=multipletests(grid.p,method="fdr_bh")[1]; grid["sig"]=grid.p_fdr<0.05
print(f"grid cells (n>=15): {len(grid)}, significant after FDR: {grid.sig.sum()}")
print("by layer (pooled):")
for l in ["2/3","4","5","6a","6b"]:
    s=Gv[Gv.layer==l]
    if len(s)>=15: print(f"  L{l}: n={len(s)}, median DvI_90={s.DvI_90.median():+.3f}")

## 4 · Population time-series by area

Mean ± SEM PSTHs, three normalizations. Because many units are near-silent at baseline, % change
and z-score gate on baseline rate/SD (bin-wise *median* of low-rate PSTHs is quantized, and dividing
by a near-zero baseline explodes — so we use **mean ± SEM with silent-unit gating**).

In [ ]:
#@title Population PSTHs by area (Hz / %-change / z-score)
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
# assemble tensor + parallel metadata across sessions
PSTH=np.concatenate([P for _,P in Ps],axis=0)
pm=pd.concat([d[["subject","area","layer","responsive"]] for d in dfs],ignore_index=True)
ci={c:i for i,c in enumerate(CONDS)}; tms=CEN*1000
PSTHs=gaussian_filter1d(PSTH,1.5,axis=2)
bm_=PSTHs[:,0,:][:,CEN<0].mean(1); bs_=PSTHs[:,0,:][:,CEN<0].std(1)
PCT=(PSTHs-bm_[:,None,None])/bm_[:,None,None]*100; Z=(PSTHs-bm_[:,None,None])/bs_[:,None,None]
resp=pm.responsive.values; gate_pct=bm_>=1.0; gate_z=bs_>=0.3
COL={"standard":"#7f7f7f","oddball_90":"#c0392b","control_90":"#2c7fb8"}
LAB={"standard":"standard","oddball_90":"90° oddball","control_90":"90° control"}
areas=["VISp","VISl","VISlm","VISa"]; norms=[("Hz",PSTHs,None),("% change",PCT,gate_pct),("z-score",Z,gate_z)]
def band(ax,T,mask,cond,c,lab):
    X=T[mask,ci[cond],:]; mu=X.mean(0); sem=X.std(0)/np.sqrt(mask.sum())
    ax.plot(tms,mu,color=c,lw=1.6,label=lab); ax.fill_between(tms,mu-sem,mu+sem,color=c,alpha=0.2,lw=0)
fig,axes=plt.subplots(3,4,figsize=(14,9),sharex=True)
for row,(ylab,T,gate) in enumerate(norms):
    for cj,a in enumerate(areas):
        ax=axes[row,cj]; mask=resp&(pm.area.values==a)&(gate if gate is not None else np.ones(len(resp),bool))
        for cd in ["standard","control_90","oddball_90"]: band(ax,T,mask,cd,COL[cd],LAB[cd] if (row==0 and cj==0) else None)
        ax.axvline(0,color="k",lw=0.5,ls=":");
        if row>0: ax.axhline(0,color="0.8",lw=0.6)
        if row==0: ax.set_title(f"{a} (n={mask.sum()})",fontsize=9)
        if cj==0: ax.set_ylabel(ylab); 
        if row==2: ax.set_xlabel("time (ms)")
        ax.set_xlim(-100,500); ax.set_xticks([0,200,400])
axes[0,0].legend(frameon=False,fontsize=7,loc="upper left")
fig.suptitle("Population deviance dynamics by area — mean ± SEM",y=0.995,fontsize=12)
fig.tight_layout(); plt.show()

## 5 · Omission prediction error — a tuning-free deviant

When an expected grating is *omitted*, a positive response is prediction error by construction
(there is no stimulus to drive it). This contrast needs no tuning control.

In [ ]:
#@title Omission response
Go=G  # omission fields already in the per-unit table
print(f"pooled omission response: median={Go.om_resp.median():+.3f} Hz, p={ss.wilcoxon(Go.om_resp).pvalue:.1e}, {np.mean(Go.om_resp>0)*100:.0f}% of units > 0")
sess=Go.groupby("subject").om_resp.median()
print(f"sessions with positive median: {(sess>0).sum()}/{len(sess)}")
print("by layer:")
for l in ["2/3","4","5","6a"]:
    s=Go[(Go.layer==l)&(Go.area.str.startswith("VIS"))]
    if len(s)>=15: print(f"  L{l}: n={len(s)}, median={s.om_resp.median():+.3f} Hz")

## 6 · Are feature-deviance and omission the same signal?

Both are measured on the *same units*. We z-score each within its error type and ask whether a unit's
feature-deviance predicts its omission response, and whether the laminar profiles differ.

In [ ]:
#@title Per-unit independence of the two error signals
zf=(G.DvI_90-G.DvI_90.mean())/G.DvI_90.std()
zo=(G.om_resp-G.om_resp.mean())/G.om_resp.std()
rho,pr=ss.spearmanr(zf,zo)
print(f"per-unit Spearman(feature DvI, omission): rho={rho:+.3f}, p={pr:.2e}")
print("=> near-zero correlation: the two error types are carried largely independently at the single-unit level")
gv=G[G.area.str.startswith("VIS")&G.layer.isin(["2/3","4","5","6a"])].copy()
gv["diff"]=(gv.DvI_90-gv.DvI_90.mean())/gv.DvI_90.std()-(gv.om_resp-gv.om_resp.mean())/gv.om_resp.std()
kw=ss.kruskal(*[gv[gv.layer==l]["diff"].values for l in ["2/3","4","5","6a"]])
print(f"laminar modulation of the balance: Kruskal H={kw[0]:.2f}, p={kw[1]:.3f} (weak; + = feature-leaning)")

## Takeaway

- **Adaptation-controlled deviance is robust and reproducible**: pooled DvI_90 ≈ +0.46, positive in
  all 9 sessions, far exceeding the adaptation-inflated naive index.
- **It is not a tuning-sampling artifact**: DvI is independent of orientation preference, present in
  tuned and untuned units alike, and unchanged by orientation-balanced resampling.
- **It has area × layer structure** but a *broadcast* character (present almost everywhere).
- **Omission** drives a positive, tuning-free prediction-error response, reproducible across sessions.
- **Feature-deviance and omission are largely independent at the single-unit level** (Spearman ρ ≈ 0.06),
  with only a weak laminar modulation of their balance — a middle position between a single canonical
  error signal and cleanly segregated circuits.

See `docs/oddball_analysis_plan.md` for the pre-registration and `README.md` for the narrative.

---
*Generated for the `ai_oscp_neuro` project. Data: DANDI 001637 (OpenScope Community Predictive
Processing, Allen Institute for Neural Dynamics). Streams anonymously — no DANDI credentials needed.*